In [1]:
# qcqp_gnn_regressor.py
#
# GNN pipeline: predicts continuous delta = (z_lp - z_sdp) / denom  (regression).
#
# DELTA COMPUTATION — aligned to the second code:
#
#   delta = (z_lp - z_sdp) / (|z_lp| + |z_sdp| + epsilon_gap)
#
# Unbounded instance handling:
#   SDP unbounded + LP finite  -> delta = +1.0  (LP wins hard)   -> included in loss
#   LP  unbounded + SDP finite -> delta = -1.0  (SDP wins hard)  -> included in loss
#   Both finite               -> delta from fixed additive gap    -> included in loss
#   Both failed / unknown     -> excluded (mask=0)
#
# Outputs saved to:
#   gnn_reg_outputs/<RUN_TAG>/
#       results_reg_<RUN_TAG>.csv
#       preds_<RUN_TAG>.csv
#       scatter_<RUN_TAG>.png
#       sign_cm_<RUN_TAG>.png/.csv
#       epoch_log_<RUN_TAG>.csv
#       config_<RUN_TAG>.csv

import os
import re
import pickle
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Subset

from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool
from torch_geometric.utils import scatter

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    confusion_matrix, ConfusionMatrixDisplay,
)


# ============================================================
# 0) Constants
# ============================================================

UNBOUNDED_STATUSES = frozenset({
    "unbounded", "unbounded_inaccurate", "unbounded_uncertified",
})

EXCLUDE_STATUSES = frozenset({
    "infeasible", "infeasible_inaccurate", "infeasible_or_unbounded",
    "primal_infeasible", "dual_infeasible_cert", "unknown",
    "numeric_error", "time_limit", "iteration_limit", "node_limit",
    "interrupted", "memory_limit",
})


# ============================================================
# 1) RunConfig
# ============================================================

@dataclass
class RunConfig:
    # Data
    train_pkl:  str           = "instances_5_10_10.pkl"
    train_csv:  Optional[str] = "instancesobj_5_10_10.csv"
    test_pkl:   Optional[str] = "instances_5_1_50.pkl"
    test_csv:   Optional[str] = "instancesobj_5_1_50.csv"
    cache_dir:  Optional[str] = None

    # Output
    output_root: str = "gnn_reg_outputs"

    # Model
    epochs:       int   = 10
    hidden:       int   = 64
    layers:       int   = 3
    dropout:      float = 0.1
    lr:           float = 1e-3
    weight_decay: float = 1e-5

    # Batching
    batch_train: int = 8
    batch_eval:  int = 16

    # Fixed additive gap formula
    epsilon_gap: float = 1e-3

    # Delta transform
    delta_transform: str = "none"

    # Sign / decision threshold
    tau_sign: float = 0.1

    # Sample weights
    # finite rows: weight from |delta_raw|
    # unbounded rows: delta_raw = NaN -> base weight only = (1-alpha)
    use_sample_weights: bool  = True
    weight_alpha:       float = 0.30

    # Graph construction
    tol:                      float = 1e-12
    max_quad_nodes:           int   = -1
    max_lin_edges_per_constr: int   = -1

    # Dataset limits
    max_items_train: int = -1
    max_items_test:  int = -1

    # Optional filters
    train_size_tag:  Optional[str] = None
    test_size_tag:   Optional[str] = None
    train_n:         Optional[int] = None
    train_m_total:   Optional[int] = None
    test_n:          Optional[int] = None
    test_m_total:    Optional[int] = None

    # Debug
    verbose:                  bool = True
    allow_row_order_fallback: bool = True


# ============================================================
# Helpers
# ============================================================

def _ts() -> str:
    return datetime.now().strftime("%H:%M:%S")


def _run_tag(cfg: RunConfig) -> str:
    def stem(p): return Path(p).stem if p else "none"
    tr = stem(cfg.train_pkl)
    te = stem(cfg.test_pkl) if cfg.test_pkl else "holdout"
    return (f"reg__train[{tr}]__test[{te}]"
            f"__h{cfg.hidden}l{cfg.layers}ep{cfg.epochs}"
            f"__gap[fixed]__xform[{cfg.delta_transform}]")


def _make_run_dir(cfg: RunConfig) -> Tuple[Path, str]:
    root = Path(cfg.output_root)
    tag  = _run_tag(cfg)
    run_dir = root / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir, tag


# ============================================================
# 2) PKL loader
# ============================================================

class RobustUnpickler(pickle.Unpickler):
    _cache = {}
    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except Exception:
            key = (module, name)
            if key not in self._cache:
                self._cache[key] = type(name, (object,), {})
            return self._cache[key]


def load_instances_pkl(pkl_path: str) -> List[Any]:
    with open(pkl_path, "rb") as f:
        data = RobustUnpickler(f).load()
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        return data["instances"] if "instances" in data else list(data.values())
    return list(data)


# ============================================================
# 3) Low-level helpers
# ============================================================

def get_field(obj, name, default=None):
    if isinstance(obj, dict): return obj.get(name, default)
    return getattr(obj, name, default)


def to_key(v) -> Optional[str]:
    if v is None: return None
    if isinstance(v, float) and np.isnan(v): return None
    return str(v)


def safe_filename(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    return s[:180]


def as_1d(x) -> np.ndarray:
    return np.asarray(x, dtype=float).reshape(-1)


def as_2d(x) -> np.ndarray:
    a = np.asarray(x, dtype=float)
    if a.ndim != 2:
        raise ValueError(f"Expected 2D array, got shape {a.shape}")
    return a


def symmetrize(A: np.ndarray) -> np.ndarray:
    return 0.5 * (A + A.T)


_FEAT_CLIP = 1e4

def _safe(arr: np.ndarray) -> np.ndarray:
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(arr, -_FEAT_CLIP, _FEAT_CLIP)


def _safe_scalar(v: float) -> float:
    if not np.isfinite(v): return 0.0
    return float(np.clip(v, -_FEAT_CLIP, _FEAT_CLIP))


def make_edge_tensors(src, dst, attr):
    if len(src) == 0:
        return (torch.empty((2, 0), dtype=torch.long),
                torch.empty((0, 1),  dtype=torch.float))
    ei = torch.tensor([src, dst], dtype=torch.long)
    ea = torch.tensor(attr, dtype=torch.float)
    if ea.dim() == 1:
        ea = ea.view(-1, 1)
    return ei, ea


# ============================================================
# 4) Status helpers
# ============================================================

def _norm_status(s) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)): return ""
    return str(s).strip().lower()


def _is_unbounded(status: str) -> bool:
    return status in UNBOUNDED_STATUSES


def _is_excluded(status: str) -> bool:
    return status in EXCLUDE_STATUSES


# ============================================================
# 5) Fixed additive delta
# ============================================================

def _apply_delta_transform(delta: float, transform: str) -> float:
    t = transform.strip().lower()
    if t == "tanh": return float(np.tanh(delta))
    if t == "clip": return float(np.clip(delta, -1.0, 1.0))
    return float(delta)


def compute_delta_fixed(z_lp: float, z_sdp: float,
                        epsilon_gap: float = 1e-3) -> float:
    denom = abs(z_lp) + abs(z_sdp) + float(epsilon_gap)
    return float((z_lp - z_sdp) / denom)


# ============================================================
# 6) Regression target creation
# ============================================================

def make_regression_target_from_row(
    row:             pd.Series,
    epsilon_gap:     float = 1e-3,
    delta_transform: str   = "none",
) -> Tuple[float, int, str, float]:
    """
    Returns (delta_target, choice_mask, target_case, delta_raw_for_weight).

    choice_mask = 1 -> included in MSE loss
    choice_mask = 0 -> excluded (both failed / unknown)

    target_case:
        "sdp_unbounded"  -> delta = +1.0  (LP wins hard)
        "lp_unbounded"   -> delta = -1.0  (SDP wins hard)
        "finite"         -> delta from fixed additive gap formula
        "excluded"       -> delta = 0.0, mask = 0

    delta_raw_for_weight:
        finite rows      -> raw finite delta before transform
        unbounded rows   -> NaN
        excluded rows    -> NaN
    """
    lp_status  = _norm_status(row.get("lp_status",  ""))
    sdp_status = _norm_status(row.get("sdp_status", ""))
    z_lp  = row.get("lp_obj",  np.nan)
    z_sdp = row.get("sdp_obj", np.nan)

    lp_fin  = (pd.notna(z_lp)  and np.isfinite(float(z_lp))
               and not _is_unbounded(lp_status)  and not _is_excluded(lp_status))
    sdp_fin = (pd.notna(z_sdp) and np.isfinite(float(z_sdp))
               and not _is_unbounded(sdp_status) and not _is_excluded(sdp_status))

    lp_unb  = (_is_unbounded(lp_status)
               or (pd.notna(z_lp)  and np.isneginf(float(z_lp))))
    sdp_unb = (_is_unbounded(sdp_status)
               or (pd.notna(z_sdp) and np.isneginf(float(z_sdp))))

    if sdp_unb and lp_fin:
        return _apply_delta_transform(+1.0, delta_transform), 1, "sdp_unbounded", np.nan

    if lp_unb and sdp_fin:
        return _apply_delta_transform(-1.0, delta_transform), 1, "lp_unbounded", np.nan

    if lp_fin and sdp_fin:
        raw   = compute_delta_fixed(float(z_lp), float(z_sdp), epsilon_gap=epsilon_gap)
        delta = _apply_delta_transform(raw, delta_transform)
        return float(delta), 1, "finite", float(raw)

    return 0.0, 0, "excluded", np.nan


# ============================================================
# 7) Term selection
# ============================================================

def upper_tri_coeffs_for_half_quadratic(A: np.ndarray) -> np.ndarray:
    n = A.shape[0]
    iu, ju = np.triu_indices(n)
    coeff  = A[iu, ju].astype(float)
    coeff[iu == ju] *= 0.5
    return coeff


def triu_flat_index(iu, ju, n) -> np.ndarray:
    iu = iu.astype(np.int64); ju = ju.astype(np.int64)
    return (iu * n - (iu * (iu - 1)) // 2 + (ju - iu)).astype(np.int64)


def select_top_pairs_across_all_matrices(A_list, tol, max_quad_nodes):
    n = A_list[0].shape[0]
    iu_all, ju_all = np.triu_indices(n)
    score = np.zeros(len(iu_all), dtype=float)
    for Ak in A_list:
        score = np.maximum(
            score,
            np.abs(upper_tri_coeffs_for_half_quadratic(symmetrize(Ak))))
    keep = score > tol
    iu = iu_all[keep].astype(int); ju = ju_all[keep].astype(int)
    if len(iu) == 0:
        return np.array([0], dtype=int), np.array([0], dtype=int)
    if max_quad_nodes is not None and len(iu) > max_quad_nodes:
        sc    = score[keep]
        idx   = np.argpartition(sc, -max_quad_nodes)[-max_quad_nodes:]
        order = np.argsort(-sc[idx])
        iu = iu[idx][order]; ju = ju[idx][order]
    return iu, ju


# ============================================================
# 8) Graph config
# ============================================================

@dataclass
class GraphConfig:
    tol:                      float         = 1e-12
    epsilon_gap:              float         = 1e-3
    delta_transform:          str           = "none"
    max_quad_nodes:           Optional[int] = None
    max_lin_edges_per_constr: Optional[int] = None


# ============================================================
# 9) Graph builder
# ============================================================

def build_tripartite_term_graph(inst, row: pd.Series, cfg: GraphConfig) -> HeteroData:
    A_list = get_field(inst, "A_list"); b_list = get_field(inst, "b_list")
    c_list = get_field(inst, "c_list"); l = get_field(inst, "l"); u = get_field(inst, "u")
    if any(x is None for x in [A_list, b_list, c_list, l, u]):
        raise ValueError("Instance missing required fields.")

    A_list = [as_2d(Ak) for Ak in A_list]
    b_list = [as_1d(bk) for bk in b_list]
    c_list = [float(np.asarray(ck).reshape(-1)[0]) for ck in c_list]
    l = as_1d(l); u = as_1d(u)
    n = int(l.shape[0]); m_total = len(A_list); m_constr = m_total - 1
    data = HeteroData()

    p0  = _safe(b_list[0].astype(float))
    l_s = _safe(l); u_s = _safe(u)
    data["var"].x = torch.tensor(
        np.stack([p0, l_s, u_s, (u_s - l_s), np.ones_like(l_s)], axis=1),
        dtype=torch.float)

    cc = _safe(np.array([_safe_scalar(c_list[k]) for k in range(1, m_total)], dtype=float))
    data["constr"].x = torch.tensor(
        np.stack([cc, np.ones_like(cc)], axis=1), dtype=torch.float)

    iu_sel, ju_sel = select_top_pairs_across_all_matrices(
        A_list, cfg.tol, cfg.max_quad_nodes)
    sel_idx    = triu_flat_index(iu_sel, ju_sel, n)
    coeff0_sel = _safe(upper_tri_coeffs_for_half_quadratic(
        symmetrize(A_list[0]))[sel_idx])
    is_diag = (iu_sel == ju_sel).astype(float)
    data["quad"].x = torch.tensor(
        np.stack([coeff0_sel, is_diag,
                  iu_sel / max(1.0, n - 1), ju_sel / max(1.0, n - 1),
                  np.ones_like(is_diag)], axis=1), dtype=torch.float)

    # var -> quad
    vq_src, vq_dst, vq_attr = [], [], []
    for q_id, (i, j) in enumerate(zip(iu_sel.tolist(), ju_sel.tolist())):
        vq_src.append(int(i)); vq_dst.append(int(q_id)); vq_attr.append([1.0])
        if int(j) != int(i):
            vq_src.append(int(j)); vq_dst.append(int(q_id)); vq_attr.append([1.0])
    ei, ea = make_edge_tensors(vq_src, vq_dst, vq_attr)
    data["var",  "inc",     "quad"].edge_index = ei
    data["var",  "inc",     "quad"].edge_attr  = ea
    rev = torch.stack([ei[1], ei[0]], 0) if ei.numel() > 0 else torch.empty((2, 0), dtype=torch.long)
    data["quad", "rev_inc", "var"].edge_index  = rev
    data["quad", "rev_inc", "var"].edge_attr   = ea

    # quad -> constr
    qc_src, qc_dst, qc_attr = [], [], []
    for k in range(1, m_total):
        coeffk = _safe(upper_tri_coeffs_for_half_quadratic(
            symmetrize(A_list[k]))[sel_idx])
        for q_id in np.where(np.abs(coeffk) > cfg.tol)[0]:
            qc_src.append(int(q_id)); qc_dst.append(int(k - 1))
            qc_attr.append([float(coeffk[q_id])])
    ei, ea = make_edge_tensors(qc_src, qc_dst, qc_attr)
    data["quad",   "qcoef",     "constr"].edge_index = ei
    data["quad",   "qcoef",     "constr"].edge_attr  = ea
    rev = torch.stack([ei[1], ei[0]], 0) if ei.numel() > 0 else torch.empty((2, 0), dtype=torch.long)
    data["constr", "rev_qcoef", "quad"].edge_index   = rev
    data["constr", "rev_qcoef", "quad"].edge_attr    = ea

    # var -> constr
    vc_src, vc_dst, vc_attr = [], [], []
    for k in range(1, m_total):
        bk   = _safe(b_list[k].astype(float))
        keep = (np.arange(n) if (cfg.max_lin_edges_per_constr is None
                                 or cfg.max_lin_edges_per_constr >= n)
                else np.argpartition(
                    np.abs(bk), -cfg.max_lin_edges_per_constr
                )[-cfg.max_lin_edges_per_constr:])
        for i in keep:
            val = float(bk[int(i)])
            if abs(val) > cfg.tol:
                vc_src.append(int(i)); vc_dst.append(int(k - 1))
                vc_attr.append([val])
    ei, ea = make_edge_tensors(vc_src, vc_dst, vc_attr)
    data["var",    "lin",     "constr"].edge_index = ei
    data["var",    "lin",     "constr"].edge_attr  = ea
    rev = torch.stack([ei[1], ei[0]], 0) if ei.numel() > 0 else torch.empty((2, 0), dtype=torch.long)
    data["constr", "rev_lin", "var"].edge_index    = rev
    data["constr", "rev_lin", "var"].edge_attr     = ea

    data.global_x = torch.tensor(
        [[_safe_scalar(c_list[0]), float(n) / 500.0, float(m_constr) / 500.0]],
        dtype=torch.float)

    delta, choice_mask, target_case, delta_raw_for_weight = make_regression_target_from_row(
        row,
        epsilon_gap=cfg.epsilon_gap,
        delta_transform=cfg.delta_transform)

    data.delta                = torch.tensor([float(delta)], dtype=torch.float)
    data.choice_mask          = torch.tensor([float(choice_mask)], dtype=torch.float)
    data.delta_raw_for_weight = torch.tensor(
        [float(np.nan_to_num(delta_raw_for_weight, nan=0.0))], dtype=torch.float
    )
    data.target_case = target_case
    data.row_key = to_key(
        row.get("id_global", row.get("id", row.get("__match_key__", "NA"))))
    return data


# ============================================================
# 10) Dataset
# ============================================================

def build_dataframe_from_instance_results(instances) -> pd.DataFrame:
    rows = []
    for inst in instances:
        res = get_field(inst, "results", {}) or {}
        if not isinstance(res, dict): res = {}
        rows.append({
            "instance_id": to_key(get_field(inst, "instance_id")),
            "group_id":    to_key(get_field(inst, "group_id")),
            "size_tag":    to_key(get_field(inst, "size_tag")),
            "id":          to_key(get_field(inst, "instance_id")),
            "lp_obj":      res.get("lp_obj",    np.nan),
            "lp_status":   res.get("lp_status",  ""),
            "sdp_obj":     res.get("sdp_obj",   np.nan),
            "sdp_status":  res.get("sdp_status", ""),
        })
    df = pd.DataFrame(rows)
    if (df[["lp_obj", "sdp_obj"]].notna().sum().sum() == 0
            and (df["lp_status"].astype(str) == "").all()):
        raise RuntimeError("No CSV and instances[].results has no LP/SDP labels.")
    return df


class QCQPTripartiteDataset(torch.utils.data.Dataset):
    def __init__(self, pkl_path, csv_path, cfg: GraphConfig,
                 cache_dir=None, max_items=None, require_labels=True,
                 filter_size_tag=None, filter_n=None, filter_m_total=None,
                 verbose=True, allow_row_order_fallback=True):
        super().__init__()
        self.cfg = cfg; self.cache_dir = cache_dir; self.verbose = verbose
        if cache_dir: os.makedirs(cache_dir, exist_ok=True)
        self.instances = load_instances_pkl(pkl_path)

        if csv_path and os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
        elif require_labels:
            df = build_dataframe_from_instance_results(self.instances)
        else:
            df = pd.DataFrame()

        self.candidate_keys = ["id_global", "id", "instance_id", "group_id", "size_tag"]
        self.instances_by_key: Dict[str, Any] = {}
        for idx, inst in enumerate(self.instances):
            for kn in self.candidate_keys:
                k = to_key(get_field(inst, kn))
                if k is not None: self.instances_by_key[k] = inst
            self.instances_by_key[f"__rowidx__{idx}"] = inst

        if df.shape[0] == 0:
            df = pd.DataFrame([{
                "__match_key__": f"__rowidx__{i}",
                "lp_obj": np.nan, "lp_status": "",
                "sdp_obj": np.nan, "sdp_status": ""}
                for i in range(len(self.instances))])

        def row_key_match(row):
            for k in self.candidate_keys:
                v = to_key(row.get(k))
                if v is not None and v in self.instances_by_key: return v
            return None

        match         = [row_key_match(row) for _, row in df.iterrows()]
        keep_mask     = [m is not None for m in match]
        matched_count = int(np.sum(keep_mask))

        if matched_count == 0 and allow_row_order_fallback:
            if len(df) == len(self.instances):
                if self.verbose:
                    print(f"[{_ts()}] [match] No key matches; ROW-ORDER fallback.")
                df = df.copy()
                df["__match_key__"] = [f"__rowidx__{i}" for i in range(len(df))]
            else:
                raise RuntimeError(
                    f"No rows matched and row-order fallback impossible "
                    f"(df={len(df)}, instances={len(self.instances)}).")
        else:
            df = df[np.array(keep_mask, dtype=bool)].copy()
            df["__match_key__"] = [m for m in match if m is not None]
            df.reset_index(drop=True, inplace=True)
            if self.verbose:
                print(f"[{_ts()}] [match] matched rows: {len(df)}/{len(match)}")

        if filter_size_tag is not None:
            df = df[df.get("size_tag", "").astype(str) == str(filter_size_tag)].copy()
            df.reset_index(drop=True, inplace=True)

        if filter_n is not None or filter_m_total is not None:
            ok_rows = []
            for i in range(len(df)):
                inst = self.instances_by_key[str(df.loc[i, "__match_key__"])]
                n_i  = int(get_field(inst, "n", -1))
                m_i  = int(get_field(inst, "m", -1))
                ok   = ((filter_n is None or n_i == filter_n)
                        and (filter_m_total is None or m_i == filter_m_total))
                ok_rows.append(ok)
            df = df[np.array(ok_rows, dtype=bool)].copy()
            df.reset_index(drop=True, inplace=True)

        dd_list, mm_list, cc_list, drw_list = [], [], [], []
        for _, row in df.iterrows():
            d, m, c, drw = make_regression_target_from_row(
                row,
                epsilon_gap=self.cfg.epsilon_gap,
                delta_transform=self.cfg.delta_transform)
            dd_list.append(d); mm_list.append(m); cc_list.append(c); drw_list.append(drw)
        df["__delta__"]                = dd_list
        df["__choice_mask__"]          = mm_list
        df["__target_case__"]          = cc_list
        df["__delta_raw_for_weight__"] = drw_list

        case_counts = pd.Series(cc_list).value_counts().to_dict()
        total       = len(cc_list)
        n_included  = int(np.sum(mm_list))
        print(f"[{_ts()}] [target] case counts (N={total}): {case_counts}")
        print(f"[{_ts()}] [target] included in loss: {n_included}/{total} "
              f"({100 * n_included / max(1, total):.1f}%)")

        if max_items and max_items > 0:
            df = df.iloc[:max_items].copy()
            df.reset_index(drop=True, inplace=True)
        self.df = df

    def __len__(self): return len(self.df)

    def _cache_path(self, idx, row):
        return os.path.join(
            self.cache_dir,
            f"graph_{idx:08d}_{safe_filename(str(row['__match_key__']))}.pt")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.cache_dir:
            cp = self._cache_path(idx, row)
            if os.path.exists(cp): return torch.load(cp)
        inst = self.instances_by_key[str(row["__match_key__"])]
        g    = build_tripartite_term_graph(inst, row, self.cfg)
        if self.cache_dir: torch.save(g, cp)
        return g


# ============================================================
# 11) Model — single regression head
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Linear(out_dim, out_dim))
    def forward(self, x): return self.net(x)


def edge_weighted_sum(x_src, edge_index, edge_attr, num_dst):
    if edge_index.numel() == 0:
        return x_src.new_zeros((num_dst, x_src.size(1)))
    msg = x_src[edge_index[0]] * edge_attr
    return scatter(msg, edge_index[1], dim=0, dim_size=num_dst, reduce="sum")


class TripartiteTermGNN(nn.Module):
    def __init__(self, hidden=64, layers=3, dropout=0.1):
        super().__init__()
        self.hidden = hidden; self.layers = layers; self.dropout = dropout
        H = hidden
        self.lin_var  = nn.Linear(5, H)
        self.lin_quad = nn.Linear(5, H)
        self.lin_con  = nn.Linear(2, H)
        self.f_v_to_q = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_v_to_c = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_q_to_c = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_c_to_q = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_c_to_v = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_q_to_v = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.g_q1 = nn.ModuleList([MLP(H * 2, H) for _ in range(layers)])
        self.g_c  = nn.ModuleList([MLP(H * 3, H) for _ in range(layers)])
        self.g_q2 = nn.ModuleList([MLP(H * 2, H) for _ in range(layers)])
        self.g_v  = nn.ModuleList([MLP(H * 3, H) for _ in range(layers)])
        head_in = H * 3 + 3
        self.head_delta = nn.Sequential(
            nn.Linear(head_in, H), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(H, 1))

    def forward(self, data):
        h_v = self.lin_var(data["var"].x)
        h_q = self.lin_quad(data["quad"].x)
        h_c = self.lin_con(data["constr"].x)
        e_vq, w_vq = data["var",   "inc",       "quad"].edge_index,  data["var",   "inc",       "quad"].edge_attr
        e_vc, w_vc = data["var",   "lin",       "constr"].edge_index, data["var",   "lin",       "constr"].edge_attr
        e_qc, w_qc = data["quad",  "qcoef",     "constr"].edge_index, data["quad",  "qcoef",     "constr"].edge_attr
        e_cq, w_cq = data["constr","rev_qcoef", "quad"].edge_index,   data["constr","rev_qcoef", "quad"].edge_attr
        e_cv, w_cv = data["constr","rev_lin",   "var"].edge_index,    data["constr","rev_lin",   "var"].edge_attr
        e_qv, w_qv = data["quad",  "rev_inc",   "var"].edge_index,    data["quad",  "rev_inc",   "var"].edge_attr
        nv = data["var"].num_nodes; nq = data["quad"].num_nodes; nc = data["constr"].num_nodes

        for t in range(self.layers):
            h_q_bar = self.g_q1[t](torch.cat([
                h_q, edge_weighted_sum(self.f_v_to_q[t](h_v), e_vq, w_vq, nq)], 1))
            h_c_new = self.g_c[t](torch.cat([
                h_c,
                edge_weighted_sum(self.f_v_to_c[t](h_v),     e_vc, w_vc, nc),
                edge_weighted_sum(self.f_q_to_c[t](h_q_bar), e_qc, w_qc, nc)], 1))
            h_q_new = self.g_q2[t](torch.cat([
                h_q_bar,
                edge_weighted_sum(self.f_c_to_q[t](h_c_new), e_cq, w_cq, nq)], 1))
            h_v_new = self.g_v[t](torch.cat([
                h_v,
                edge_weighted_sum(self.f_c_to_v[t](h_c_new), e_cv, w_cv, nv),
                edge_weighted_sum(self.f_q_to_v[t](h_q_new), e_qv, w_qv, nv)], 1))
            h_v = F.dropout(F.relu(h_v_new), p=self.dropout, training=self.training)
            h_q = F.dropout(F.relu(h_q_new), p=self.dropout, training=self.training)
            h_c = F.dropout(F.relu(h_c_new), p=self.dropout, training=self.training)

        B  = int(data["var"].batch.max().item()) + 1
        pv = global_mean_pool(h_v, data["var"].batch, size=B)
        pq = (global_mean_pool(h_q, data["quad"].batch, size=B)
              if nq > 0 else h_v.new_zeros((B, self.hidden)))
        pc = (global_mean_pool(h_c, data["constr"].batch, size=B)
              if nc > 0 else h_v.new_zeros((B, self.hidden)))
        gx = data.global_x
        if gx.dim() == 1: gx = gx.view(-1, 3)
        if gx.size(0) != B: gx = gx.view(B, -1)
        g = torch.cat([pv, pq, pc, gx], dim=1)
        return self.head_delta(g).view(-1)


# ============================================================
# 12) Metrics
# ============================================================

def safe_pearson_r(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    yt, yp = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    if np.std(yt) <= 0 or np.std(yp) <= 0: return 0.0
    r = np.corrcoef(yt, yp)[0, 1]
    return float(r) if np.isfinite(r) else 0.0


def r_accuracy(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 0.1) -> float:
    y    = np.asarray(y_true, dtype=float)
    yhat = np.asarray(y_pred, dtype=float)
    m    = np.isfinite(y) & np.isfinite(yhat)
    y, yhat = y[m], yhat[m]
    return float(np.mean(
        (np.sign(y) == np.sign(yhat)) | ((y > -eps) & (y < 0) & (yhat > 0))))


def sign_accuracy_tau(y_true: np.ndarray, y_pred: np.ndarray,
                      tau: float = 0.1) -> Tuple[float, float]:
    yt, yp = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    mask   = np.isfinite(yt) & np.isfinite(yp) & (np.abs(yt) >= tau)
    if mask.sum() == 0: return 0.0, 0.0
    return (float(np.mean(np.sign(yt[mask]) == np.sign(yp[mask]))),
            float(mask.mean()))


# ============================================================
# 13) Sample weights
# ============================================================

def make_sample_weight_tensor(delta_raw_for_weight: torch.Tensor, alpha: float) -> torch.Tensor:
    a = float(np.clip(alpha, 0.0, 1.0))
    strength = torch.nan_to_num(delta_raw_for_weight.abs(), nan=0.0, posinf=0.0, neginf=0.0)
    return (1.0 - a) + a * strength


# ============================================================
# 14) Train / Eval
# ============================================================

def train_one_epoch(model, loader, opt, device,
                    use_sample_weights: bool  = True,
                    weight_alpha:       float = 0.30) -> float:
    model.train()
    total_loss = 0.0; total_n = 0

    for batch in loader:
        batch = batch.to(device); opt.zero_grad()
        lk = model(batch)
        d  = batch.delta.view(-1).float()
        m  = batch.choice_mask.view(-1).float()
        drw = batch.delta_raw_for_weight.view(-1).float()

        lk_safe  = torch.where(torch.isfinite(lk), lk, torch.zeros_like(lk))
        n_active = m.sum()
        if n_active == 0: continue

        if use_sample_weights:
            w    = make_sample_weight_tensor(drw, weight_alpha)
            loss = ((lk_safe - d) ** 2 * m * w).sum() / ((m * w).sum() + 1e-12)
        else:
            loss = ((lk_safe - d) ** 2 * m).sum() / (n_active + 1e-12)

        if not torch.isfinite(loss):
            opt.zero_grad(); continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total_loss += float(loss.item()) * int(n_active.item())
        total_n    += int(n_active.item())

    return total_loss / max(1, total_n)


@torch.no_grad()
def evaluate_full(model, loader, device,
                  tau_sign: float = 0.1) -> Tuple[dict, dict, np.ndarray, np.ndarray]:
    model.eval()
    d_all, dk_all, m_all, case_all, keys_all = [], [], [], [], []

    for batch in loader:
        batch   = batch.to(device)
        lk      = model(batch)
        d       = batch.delta.view(-1).float()
        m       = batch.choice_mask.view(-1).float()
        lk_safe = torch.where(torch.isfinite(lk), lk, torch.zeros_like(lk))

        d_all.append(d.cpu()); dk_all.append(lk_safe.cpu()); m_all.append(m.cpu())

        if hasattr(batch, "target_case"):
            tc = batch.target_case
            case_all.extend(
                tc if isinstance(tc, (list, tuple)) else [tc] * int(d.numel()))
        else:
            case_all.extend(["unknown"] * int(d.numel()))

        if hasattr(batch, "row_key"):
            rk = batch.row_key
            keys_all.extend(
                [str(k) for k in rk] if isinstance(rk, (list, tuple))
                else [str(rk)] * int(d.numel()))
        else:
            keys_all.extend(["NA"] * int(d.numel()))

    d  = torch.cat(d_all).numpy()
    dk = torch.cat(dk_all).numpy()
    m  = torch.cat(m_all).numpy()

    active    = m == 1
    d_active  = d[active]; dk_active = dk[active]
    case_arr  = np.array(case_all)

    if active.sum() == 0:
        metrics = {k: float("nan") for k in [
            "mse", "mae", "pearson_r", "r_accuracy",
            "sign_acc_tau", "sign_coverage", "coverage_active"]}
        metrics["coverage_active"] = 0.0
    else:
        sacc, scov = sign_accuracy_tau(d_active, dk_active, tau=tau_sign)
        metrics = {
            "mse":             float(mean_squared_error(d_active, dk_active)),
            "mae":             float(mean_absolute_error(d_active, dk_active)),
            "pearson_r":       safe_pearson_r(d_active, dk_active),
            "r_accuracy":      r_accuracy(d_active, dk_active),
            "sign_acc_tau":    sacc,
            "sign_coverage":   scov,
            "coverage_active": float(m.mean()),
        }

    for case in ["finite", "sdp_unbounded", "lp_unbounded", "excluded"]:
        cidx            = (case_arr == case) & active
        metrics[f"n_{case}"] = int(cidx.sum())
        if cidx.sum() > 0:
            metrics[f"mse_{case}"] = float(
                mean_squared_error(d[cidx], dk[cidx]))

    preds = {
        "row_key":     keys_all,
        "delta_true":  d.tolist(),
        "delta_pred":  dk.tolist(),
        "choice_mask": m.tolist(),
        "target_case": case_all,
    }
    return metrics, preds, d_active, dk_active


# ============================================================
# 15) Deterministic split
# ============================================================

def make_deterministic_splits(keys: pd.Series, train: float = 0.8, val: float = 0.1):
    h  = pd.util.hash_pandas_object(keys.astype(str), index=False).to_numpy(dtype=np.uint64)
    b  = (h % 100).astype(np.int64)
    tc = int(train * 100); vc = int((train + val) * 100)
    return (np.where(b < tc)[0],
            np.where((b >= tc) & (b < vc))[0],
            np.where(b >= vc)[0])


def pick_split_key(df: pd.DataFrame) -> pd.Series:
    for col in ["id_global", "id", "instance_id", "group_id", "size_tag", "__match_key__"]:
        if col in df.columns and df[col].notna().any():
            return df[col].astype(str)
    return pd.Series(np.arange(len(df)), index=df.index).astype(str)


# ============================================================
# 16) Output helpers
# ============================================================

def save_scatter(d_true, d_pred, run_dir, fname, title):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(d_true, d_pred, s=10, alpha=0.45, edgecolors="none")
    lo = min(d_true.min(), d_pred.min()); hi = max(d_true.max(), d_pred.max())
    ax.plot([lo, hi], [lo, hi], "r--", label="Perfect")
    ax.axhline(0, ls="--", color="gray", alpha=0.4)
    ax.axvline(0, ls="--", color="gray", alpha=0.4)
    ax.set_xlabel("True delta"); ax.set_ylabel("Predicted delta")
    ax.set_title(title); ax.legend(); plt.tight_layout()
    plt.savefig(run_dir / fname, dpi=200); plt.close()
    print(f"[{_ts()}] [save] {fname}")


def delta_to_lp_decision(delta: np.ndarray, tau: float) -> np.ndarray:
    x = np.asarray(delta, dtype=float)
    return np.where(np.abs(x) < tau, 1, np.where(x > 0, 1, 0)).astype(int)


def save_sign_confusion_matrix(d_true, d_pred, tau, run_dir, fname, title):
    yt = np.asarray(d_true, dtype=float); yp = np.asarray(d_pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[mask], yp[mask]

    true_dec = delta_to_lp_decision(yt, tau)
    pred_dec = delta_to_lp_decision(yp, tau)

    cm  = confusion_matrix(true_dec, pred_dec, labels=[0, 1])
    acc = float(np.mean(true_dec == pred_dec))

    pd.DataFrame(cm,
        index=["true_SDP(0)", "true_LP(1)"],
        columns=["pred_SDP(0)", "pred_LP(1)"]).to_csv(
        run_dir / fname.replace(".png", ".csv"), index=True)

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=["SDP(0)", "LP(1)"]).plot(ax=ax)
    ax.set_title(f"{title}  acc={acc:.3f}  tau={tau:.3f}")
    plt.tight_layout(); plt.savefig(run_dir / fname, dpi=200); plt.close()
    print(f"[{_ts()}] [save] {fname}")


def save_epoch_log(epoch_log, run_dir, tag):
    pd.DataFrame(epoch_log).to_csv(run_dir / f"epoch_log_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] epoch_log_{tag}.csv")


def save_results(metrics, run_dir, tag, split, cfg):
    row = {"split": split, "run_tag": tag, **metrics,
           "train_pkl": cfg.train_pkl, "test_pkl": cfg.test_pkl or "holdout",
           "epochs": cfg.epochs, "hidden": cfg.hidden, "layers": cfg.layers,
           "dropout": cfg.dropout, "lr": cfg.lr,
           "epsilon_gap": cfg.epsilon_gap,
           "delta_transform": cfg.delta_transform, "tau_sign": cfg.tau_sign,
           "use_sample_weights": cfg.use_sample_weights,
           "weight_alpha": cfg.weight_alpha,
           "gap_method": "fixed_additive"}
    pd.DataFrame([row]).to_csv(run_dir / f"results_reg_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] results_reg_{tag}.csv")


def save_predictions(preds, run_dir, tag, split):
    df = pd.DataFrame(preds); df.insert(0, "split", split)
    df.to_csv(run_dir / f"preds_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] preds_{tag}.csv")


def save_config(cfg, run_dir, tag):
    pd.DataFrame([asdict(cfg)]).to_csv(run_dir / f"config_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] config_{tag}.csv")


# ============================================================
# 17) main()
# ============================================================

def main(run_cfg: Optional[RunConfig] = None):
    if run_cfg is None:
        run_cfg = RunConfig()

    np.random.seed(0); torch.manual_seed(0)
    run_dir, tag = _make_run_dir(run_cfg)

    print(f"[{_ts()}] ══════════════════════════════════════════════════")
    print(f"[{_ts()}] GNN REGRESSOR  tag={tag}")
    print(f"[{_ts()}] Output dir   : {run_dir.resolve()}")
    print(f"[{_ts()}] Gap formula  : FIXED ADDITIVE")
    print(f"[{_ts()}] delta = (lp - sdp) / (|lp| + |sdp| + epsilon_gap)")
    print(f"[{_ts()}] epsilon_gap={run_cfg.epsilon_gap}")
    print(f"[{_ts()}] delta_transform={run_cfg.delta_transform}  "
          f"tau_sign={run_cfg.tau_sign}")
    print(f"[{_ts()}] use_sample_weights={run_cfg.use_sample_weights}  "
          f"weight_alpha={run_cfg.weight_alpha}")
    print(f"[{_ts()}] Unbounded handling: sdp_unb->+1.0, lp_unb->-1.0 "
          f"(included in loss)")
    print(f"[{_ts()}] ══════════════════════════════════════════════════")

    gcfg = GraphConfig(
        tol=run_cfg.tol,
        epsilon_gap=run_cfg.epsilon_gap,
        delta_transform=run_cfg.delta_transform,
        max_quad_nodes=(None if run_cfg.max_quad_nodes <= 0
                        else int(run_cfg.max_quad_nodes)),
        max_lin_edges_per_constr=(None if run_cfg.max_lin_edges_per_constr <= 0
                                  else int(run_cfg.max_lin_edges_per_constr)),
    )
    mit = None if run_cfg.max_items_train <= 0 else int(run_cfg.max_items_train)
    mie = None if run_cfg.max_items_test  <= 0 else int(run_cfg.max_items_test)

    train_ds_full = QCQPTripartiteDataset(
        run_cfg.train_pkl, run_cfg.train_csv, gcfg,
        cache_dir=run_cfg.cache_dir, max_items=mit, require_labels=True,
        filter_size_tag=run_cfg.train_size_tag,
        filter_n=run_cfg.train_n, filter_m_total=run_cfg.train_m_total,
        verbose=run_cfg.verbose,
        allow_row_order_fallback=run_cfg.allow_row_order_fallback)

    if len(train_ds_full) == 0:
        raise RuntimeError("Train dataset empty.")

    if run_cfg.test_pkl:
        tr_idx, va_idx, _ = make_deterministic_splits(
            pick_split_key(train_ds_full.df), 0.9, 0.1)
        test_split_name = "external_test"
        use_external    = True
    else:
        tr_idx, va_idx, te_idx = make_deterministic_splits(
            pick_split_key(train_ds_full.df), 0.8, 0.1)
        test_idx_20     = np.concatenate([va_idx, te_idx])
        test_split_name = "holdout_test"
        use_external    = False

    if use_external:
        test_ds = QCQPTripartiteDataset(
            run_cfg.test_pkl, run_cfg.test_csv, gcfg,
            cache_dir=None, max_items=mie, require_labels=True,
            filter_size_tag=run_cfg.test_size_tag,
            filter_n=run_cfg.test_n, filter_m_total=run_cfg.test_m_total,
            verbose=run_cfg.verbose,
            allow_row_order_fallback=run_cfg.allow_row_order_fallback)
        if len(test_ds) == 0:
            raise RuntimeError("Test dataset empty.")
    else:
        test_ds = Subset(train_ds_full, test_idx_20.tolist())

    train_ds = Subset(train_ds_full, tr_idx.tolist())
    val_ds   = Subset(train_ds_full, va_idx.tolist())
    print(f"[{_ts()}] [data] train={len(train_ds)} | val={len(val_ds)} | test={len(test_ds)}")

    train_loader = DataLoader(train_ds, batch_size=run_cfg.batch_train, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=run_cfg.batch_eval,  shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=run_cfg.batch_eval,  shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[{_ts()}] [device] {device}")

    model = TripartiteTermGNN(run_cfg.hidden, run_cfg.layers, run_cfg.dropout).to(device)
    opt   = torch.optim.Adam(
        model.parameters(), lr=run_cfg.lr, weight_decay=run_cfg.weight_decay)

    epoch_log = []
    for epoch in range(1, run_cfg.epochs + 1):
        loss = train_one_epoch(
            model, train_loader, opt, device,
            use_sample_weights=run_cfg.use_sample_weights,
            weight_alpha=run_cfg.weight_alpha)
        vm, _, _, _ = evaluate_full(model, val_loader, device, tau_sign=run_cfg.tau_sign)
        row = {"epoch": epoch, "train_loss": loss,
               **{f"val_{k}": v for k, v in vm.items()}}
        epoch_log.append(row)
        print(f"[{_ts()}] Epoch {epoch:02d} | loss={loss:.6f} | "
              f"mse={vm['mse']:.6f} | mae={vm['mae']:.4f} | "
              f"r={vm['pearson_r']:.3f} | "
              f"sign@tau={vm['sign_acc_tau']:.3f} (cov={vm['sign_coverage']:.2f}) | "
              f"n_sdp_unb={vm.get('n_sdp_unbounded', 0)} "
              f"n_lp_unb={vm.get('n_lp_unbounded', 0)}", flush=True)

    save_epoch_log(epoch_log, run_dir, tag)

    test_metrics, preds, d_true, d_pred = evaluate_full(
        model, test_loader, device, tau_sign=run_cfg.tau_sign)
    print(f"\n[{_ts()}] TEST METRICS:")
    for k, v in test_metrics.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    save_predictions(preds, run_dir, tag, test_split_name)
    save_scatter(d_true, d_pred, run_dir, f"scatter_{tag}.png",
                 "GNN Regressor — actual vs predicted delta")
    save_sign_confusion_matrix(d_true, d_pred, run_cfg.tau_sign,
                               run_dir, f"sign_cm_{tag}.png",
                               "GNN Regression -> LP/SDP decision")
    save_results(test_metrics, run_dir, tag, test_split_name, run_cfg)
    save_config(run_cfg, run_dir, tag)

    g0 = train_ds_full[0]
    print(f"\n[{_ts()}] [graph sanity]")
    print(f"  var    : {g0['var'].num_nodes} nodes  shape={list(g0['var'].x.shape)}")
    print(f"  quad   : {g0['quad'].num_nodes} nodes  shape={list(g0['quad'].x.shape)}")
    print(f"  constr : {g0['constr'].num_nodes} nodes  shape={list(g0['constr'].x.shape)}")
    for et in g0.edge_types:
        print(f"  {et}: E={g0[et].edge_index.shape[1]}  "
              f"attr={list(g0[et].edge_attr.shape)}")
    print(f"  delta={float(g0.delta):.4f}  "
          f"choice_mask={int(g0.choice_mask)}  "
          f"target_case={g0.target_case}  "
          f"delta_raw_for_weight={float(g0.delta_raw_for_weight):.4f}")

    written = sorted(p.name for p in run_dir.glob("*"))
    print(f"\n[{_ts()}] Files written ({len(written)}): {written}")
    return model, test_metrics


if __name__ == "__main__":
    cfg = RunConfig(
        delta_transform="none",
        tau_sign=0.1,
        use_sample_weights=True,
        weight_alpha=0.30,
    )
    model, metrics = main(cfg)

[00:05:42] ══════════════════════════════════════════
[00:05:42] GNN CLASSIFIER  tag=gnnclf__train[instances_5_10_10]__test[instances_5_1_50]__h64l3ep10__gap[fixed]
[00:05:42] Output dir: C:\Users\suuser\Desktop\minlplib\gnn_outputs\gnnclf__train[instances_5_10_10]__test[instances_5_1_50]__h64l3ep10__gap[fixed]
[00:05:42] gap formula: fixed additive
[00:05:42] delta = (lp-sdp) / (|lp|+|sdp|+epsilon_gap)
[00:05:42] epsilon_gap=0.001
[00:05:42] tau_sign=0.1  eps_prime=0.1
[00:05:42] ══════════════════════════════════════════
[00:05:43] [match] matched rows: 10000/10000
[00:05:48] [match] matched rows: 50000/50000
[00:05:50] [data] train=8924 | val=1076 | test=50000
[00:05:50] [device] cpu
[00:05:50] [weights] pw_direct=0.5569
[00:07:00] Epoch 01 | loss=0.3289 | acc=0.9201 | bal_acc=0.9274 | f1_macro=0.9151 | delta_mse=0.056140 | cov=0.596
[00:08:09] Epoch 02 | loss=0.1897 | acc=0.9033 | bal_acc=0.9226 | f1_macro=0.8994 | delta_mse=0.048869 | cov=0.596
[00:09:18] Epoch 03 | loss=0.1673 | 

In [2]:
cfg = RunConfig(
    train_pkl="instances_5_2_50.pkl",
    train_csv="instancesobj_5_2_50.csv",
    test_pkl="instances_5_10_10.pkl",
    test_csv="instancesobj_5_10_10.csv",
    epochs=10,
)
model, metrics = main(cfg)

[00:19:23] ══════════════════════════════════════════
[00:19:23] GNN CLASSIFIER  tag=gnnclf__train[instances_5_2_50]__test[instances_5_10_10]__h64l3ep10__gap[fixed]
[00:19:23] Output dir: C:\Users\suuser\Desktop\minlplib\gnn_outputs\gnnclf__train[instances_5_2_50]__test[instances_5_10_10]__h64l3ep10__gap[fixed]
[00:19:23] gap formula: fixed additive
[00:19:23] delta = (lp-sdp) / (|lp|+|sdp|+epsilon_gap)
[00:19:23] epsilon_gap=0.001
[00:19:23] tau_sign=0.1  eps_prime=0.1
[00:19:23] ══════════════════════════════════════════
[00:19:28] [match] matched rows: 50000/50000
[00:19:32] [match] matched rows: 10000/10000
[00:19:32] [data] train=45062 | val=4938 | test=10000
[00:19:32] [device] cpu
[00:19:32] [weights] pw_direct=0.4429
[00:23:57] Epoch 01 | loss=0.9212 | acc=0.9433 | bal_acc=0.9112 | f1_macro=0.9298 | delta_mse=0.049259 | cov=0.542
[00:28:21] Epoch 02 | loss=0.2088 | acc=0.9350 | bal_acc=0.9258 | f1_macro=0.9237 | delta_mse=0.047277 | cov=0.542
[00:32:46] Epoch 03 | loss=0.1439 |

KeyboardInterrupt: 

In [ ]:
# qcqp_gnn_regressor_keep_unbounded.py
#
# FIXES / ALIGNMENT
#
#   FIX 1 | OPTIMAL_STATUSES kept for completeness.
#
#   FIX 2 | UNBOUNDED HANDLING — SAME LOGIC AS ML REGRESSOR KEEP-UNBOUNDED VERSION:
#            - SDP unbounded + LP finite  -> target delta = +1.0
#            - LP  unbounded + SDP finite -> target delta = -1.0
#            - both finite                -> target delta from fixed additive gap
#            - both failed / excluded     -> removed from loss/eval
#
#            IMPORTANT:
#            - unbounded rows are KEPT in the regression dataset
#            - unbounded rows ARE INCLUDED in training loss
#            - unbounded rows ARE EXCLUDED from finite-gap MSE / MAE / Pearson
#            - r_accuracy / sign_accuracy stay LP-biased on ALL active rows
#
#   FIX 3 | r_accuracy — LP-biased:
#            decision = (delta > -eps)
#
#   FIX 4 | sign_accuracy_tau — LP-biased:
#            decision = (delta > -tau)
#
#   FIX 5 | SPLIT — holdout uses 72/8/20 (train/val/test), same 20% test as ML.
#
#   FIX 6 | SAMPLE WEIGHTS — same logic everywhere:
#            w_i = (1 - alpha) + alpha * |y_i|
#            so unbounded rows with y = ±1 get full weight 1.0

import os
import re
import pickle
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Subset

from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool
from torch_geometric.utils import scatter

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


# ============================================================
# 0) Constants
# ============================================================

OPTIMAL_STATUSES = frozenset({
    "optimal",
    "optimal_inaccurate",
})

UNBOUNDED_STATUSES = frozenset({
    "unbounded",
    "unbounded_inaccurate",
    "unbounded_uncertified",
})

EXCLUDE_STATUSES = frozenset({
    "infeasible",
    "infeasible_inaccurate",
    "infeasible_or_unbounded",
    "primal_infeasible",
    "dual_infeasible_cert",
    "unknown",
    "numeric_error",
    "time_limit",
    "iteration_limit",
    "node_limit",
    "interrupted",
    "memory_limit",
})


# ============================================================
# 1) RunConfig
# ============================================================

@dataclass
class RunConfig:
    # Data
    train_pkl: str = "instances_5_10_10.pkl"
    train_csv: Optional[str] = "instancesobj_5_10_10.csv"
    test_pkl: Optional[str] = "instances_5_1_50.pkl"
    test_csv: Optional[str] = "instancesobj_5_1_50.csv"
    cache_dir: Optional[str] = None

    # Output
    output_root: str = "gnn_reg_outputs"

    # Model
    epochs: int = 10
    hidden: int = 64
    layers: int = 3
    dropout: float = 0.1
    lr: float = 1e-3
    weight_decay: float = 1e-5

    # Batching
    batch_train: int = 8
    batch_eval: int = 16

    # Fixed additive gap formula
    epsilon_gap: float = 1e-3

    # Delta transform
    delta_transform: str = "none"

    # LP-biased threshold
    tau_sign: float = 0.1

    # Sample weights
    use_sample_weights: bool = True
    weight_alpha: float = 0.30

    # Graph construction
    tol: float = 1e-12
    max_quad_nodes: int = -1
    max_lin_edges_per_constr: int = -1

    # Dataset limits
    max_items_train: int = -1
    max_items_test: int = -1

    # Optional filters
    train_size_tag: Optional[str] = None
    test_size_tag: Optional[str] = None
    train_n: Optional[int] = None
    train_m_total: Optional[int] = None
    test_n: Optional[int] = None
    test_m_total: Optional[int] = None

    # Debug
    verbose: bool = True
    allow_row_order_fallback: bool = True


# ============================================================
# Helpers
# ============================================================

def _ts() -> str:
    return datetime.now().strftime("%H:%M:%S")


def _run_tag(cfg: RunConfig) -> str:
    def stem(p):
        return Path(p).stem if p else "none"
    tr = stem(cfg.train_pkl)
    te = stem(cfg.test_pkl) if cfg.test_pkl else "holdout"
    return (
        f"reg__train[{tr}]__test[{te}]"
        f"__h{cfg.hidden}l{cfg.layers}ep{cfg.epochs}"
        f"__gap[fixed]__xform[{cfg.delta_transform}]__keepunb"
    )


def _make_run_dir(cfg: RunConfig) -> Tuple[Path, str]:
    root = Path(cfg.output_root)
    tag = _run_tag(cfg)
    run_dir = root / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir, tag


# ============================================================
# 2) PKL loader
# ============================================================

class RobustUnpickler(pickle.Unpickler):
    _cache = {}

    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except Exception:
            key = (module, name)
            if key not in self._cache:
                self._cache[key] = type(name, (object,), {})
            return self._cache[key]


def load_instances_pkl(pkl_path: str) -> List[Any]:
    with open(pkl_path, "rb") as f:
        data = RobustUnpickler(f).load()
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        return data["instances"] if "instances" in data else list(data.values())
    return list(data)


# ============================================================
# 3) Low-level helpers
# ============================================================

def get_field(obj, name, default=None):
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


def to_key(v) -> Optional[str]:
    if v is None:
        return None
    if isinstance(v, float) and np.isnan(v):
        return None
    return str(v)


def safe_filename(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    return s[:180]


def as_1d(x) -> np.ndarray:
    return np.asarray(x, dtype=float).reshape(-1)


def as_2d(x) -> np.ndarray:
    a = np.asarray(x, dtype=float)
    if a.ndim != 2:
        raise ValueError(f"Expected 2D array, got shape {a.shape}")
    return a


def symmetrize(A: np.ndarray) -> np.ndarray:
    return 0.5 * (A + A.T)


_FEAT_CLIP = 1e4

def _safe(arr: np.ndarray) -> np.ndarray:
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(arr, -_FEAT_CLIP, _FEAT_CLIP)


def _safe_scalar(v: float) -> float:
    if not np.isfinite(v):
        return 0.0
    return float(np.clip(v, -_FEAT_CLIP, _FEAT_CLIP))


def make_edge_tensors(src, dst, attr):
    if len(src) == 0:
        return (
            torch.empty((2, 0), dtype=torch.long),
            torch.empty((0, 1), dtype=torch.float),
        )
    ei = torch.tensor([src, dst], dtype=torch.long)
    ea = torch.tensor(attr, dtype=torch.float)
    if ea.dim() == 1:
        ea = ea.view(-1, 1)
    return ei, ea


# ============================================================
# 4) Status helpers
# ============================================================

def _norm_status(s) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    return str(s).strip().lower()


def _is_unbounded(status: str) -> bool:
    return status in UNBOUNDED_STATUSES


def _is_excluded(status: str) -> bool:
    return status in EXCLUDE_STATUSES


# ============================================================
# 5) Fixed additive delta
# ============================================================

def _apply_delta_transform(delta: float, transform: str) -> float:
    t = transform.strip().lower()
    if t == "tanh":
        return float(np.tanh(delta))
    if t == "clip":
        return float(np.clip(delta, -1.0, 1.0))
    return float(delta)


def compute_delta_fixed(z_lp: float, z_sdp: float, epsilon_gap: float = 1e-3) -> float:
    denom = abs(z_lp) + abs(z_sdp) + float(epsilon_gap)
    return float((z_lp - z_sdp) / denom)


# ============================================================
# 6) Regression target creation
# ============================================================

def make_regression_target_from_row(
    row: pd.Series,
    epsilon_gap: float = 1e-3,
    delta_transform: str = "none",
) -> Tuple[float, int, int, str, float]:
    """
    Returns:
        (delta_target, choice_mask, finite_mask, target_case, delta_for_weight)

    choice_mask:
        1 -> row is active in training / decision metrics
        0 -> row excluded completely

    finite_mask:
        1 -> row contributes to finite-gap MSE/MAE/Pearson
        0 -> row does NOT contribute to finite-gap MSE/MAE/Pearson

    target_case:
        "finite"
        "sdp_unbounded"
        "lp_unbounded"
        "excluded"

    delta_for_weight:
        same logical target used for sample weights:
        w = (1-alpha) + alpha*|target|
    """
    lp_status = _norm_status(row.get("lp_status", ""))
    sdp_status = _norm_status(row.get("sdp_status", ""))
    z_lp = row.get("lp_obj", np.nan)
    z_sdp = row.get("sdp_obj", np.nan)

    lp_fin = (
        pd.notna(z_lp)
        and np.isfinite(float(z_lp))
        and not _is_unbounded(lp_status)
        and not _is_excluded(lp_status)
    )
    sdp_fin = (
        pd.notna(z_sdp)
        and np.isfinite(float(z_sdp))
        and not _is_unbounded(sdp_status)
        and not _is_excluded(sdp_status)
    )

    lp_unb = _is_unbounded(lp_status) or (pd.notna(z_lp) and np.isneginf(float(z_lp)))
    sdp_unb = _is_unbounded(sdp_status) or (pd.notna(z_sdp) and np.isneginf(float(z_sdp)))

    if sdp_unb and lp_fin:
        y = _apply_delta_transform(+1.0, delta_transform)
        return float(y), 1, 0, "sdp_unbounded", float(y)

    if lp_unb and sdp_fin:
        y = _apply_delta_transform(-1.0, delta_transform)
        return float(y), 1, 0, "lp_unbounded", float(y)

    if lp_fin and sdp_fin:
        raw = compute_delta_fixed(float(z_lp), float(z_sdp), epsilon_gap=epsilon_gap)
        y = _apply_delta_transform(raw, delta_transform)
        return float(y), 1, 1, "finite", float(y)

    return 0.0, 0, 0, "excluded", np.nan


# ============================================================
# 7) Term selection
# ============================================================

def upper_tri_coeffs_for_half_quadratic(A: np.ndarray) -> np.ndarray:
    n = A.shape[0]
    iu, ju = np.triu_indices(n)
    coeff = A[iu, ju].astype(float)
    coeff[iu == ju] *= 0.5
    return coeff


def triu_flat_index(iu, ju, n) -> np.ndarray:
    iu = iu.astype(np.int64)
    ju = ju.astype(np.int64)
    return (iu * n - (iu * (iu - 1)) // 2 + (ju - iu)).astype(np.int64)


def select_top_pairs_across_all_matrices(A_list, tol, max_quad_nodes):
    n = A_list[0].shape[0]
    iu_all, ju_all = np.triu_indices(n)
    score = np.zeros(len(iu_all), dtype=float)
    for Ak in A_list:
        score = np.maximum(score, np.abs(upper_tri_coeffs_for_half_quadratic(symmetrize(Ak))))
    keep = score > tol
    iu = iu_all[keep].astype(int)
    ju = ju_all[keep].astype(int)
    if len(iu) == 0:
        return np.array([0], dtype=int), np.array([0], dtype=int)
    if max_quad_nodes is not None and len(iu) > max_quad_nodes:
        sc = score[keep]
        idx = np.argpartition(sc, -max_quad_nodes)[-max_quad_nodes:]
        order = np.argsort(-sc[idx])
        iu = iu[idx][order]
        ju = ju[idx][order]
    return iu, ju


# ============================================================
# 8) Graph config
# ============================================================

@dataclass
class GraphConfig:
    tol: float = 1e-12
    epsilon_gap: float = 1e-3
    delta_transform: str = "none"
    max_quad_nodes: Optional[int] = None
    max_lin_edges_per_constr: Optional[int] = None


# ============================================================
# 9) Graph builder
# ============================================================

def build_tripartite_term_graph(inst, row: pd.Series, cfg: GraphConfig) -> HeteroData:
    A_list = get_field(inst, "A_list")
    b_list = get_field(inst, "b_list")
    c_list = get_field(inst, "c_list")
    l = get_field(inst, "l")
    u = get_field(inst, "u")
    if any(x is None for x in [A_list, b_list, c_list, l, u]):
        raise ValueError("Instance missing required fields.")

    A_list = [as_2d(Ak) for Ak in A_list]
    b_list = [as_1d(bk) for bk in b_list]
    c_list = [float(np.asarray(ck).reshape(-1)[0]) for ck in c_list]
    l = as_1d(l)
    u = as_1d(u)

    n = int(l.shape[0])
    m_total = len(A_list)
    m_constr = m_total - 1

    data = HeteroData()

    p0 = _safe(b_list[0].astype(float))
    l_s = _safe(l)
    u_s = _safe(u)
    data["var"].x = torch.tensor(
        np.stack([p0, l_s, u_s, (u_s - l_s), np.ones_like(l_s)], axis=1),
        dtype=torch.float,
    )

    cc = _safe(np.array([_safe_scalar(c_list[k]) for k in range(1, m_total)], dtype=float))
    data["constr"].x = torch.tensor(
        np.stack([cc, np.ones_like(cc)], axis=1),
        dtype=torch.float,
    )

    iu_sel, ju_sel = select_top_pairs_across_all_matrices(A_list, cfg.tol, cfg.max_quad_nodes)
    sel_idx = triu_flat_index(iu_sel, ju_sel, n)
    coeff0_sel = _safe(upper_tri_coeffs_for_half_quadratic(symmetrize(A_list[0]))[sel_idx])
    is_diag = (iu_sel == ju_sel).astype(float)
    data["quad"].x = torch.tensor(
        np.stack([
            coeff0_sel,
            is_diag,
            iu_sel / max(1.0, n - 1),
            ju_sel / max(1.0, n - 1),
            np.ones_like(is_diag),
        ], axis=1),
        dtype=torch.float,
    )

    # var -> quad
    vq_src, vq_dst, vq_attr = [], [], []
    for q_id, (i, j) in enumerate(zip(iu_sel.tolist(), ju_sel.tolist())):
        vq_src.append(int(i))
        vq_dst.append(int(q_id))
        vq_attr.append([1.0])
        if int(j) != int(i):
            vq_src.append(int(j))
            vq_dst.append(int(q_id))
            vq_attr.append([1.0])

    ei, ea = make_edge_tensors(vq_src, vq_dst, vq_attr)
    data["var", "inc", "quad"].edge_index = ei
    data["var", "inc", "quad"].edge_attr = ea
    rev = torch.stack([ei[1], ei[0]], 0) if ei.numel() > 0 else torch.empty((2, 0), dtype=torch.long)
    data["quad", "rev_inc", "var"].edge_index = rev
    data["quad", "rev_inc", "var"].edge_attr = ea

    # quad -> constr
    qc_src, qc_dst, qc_attr = [], [], []
    for k in range(1, m_total):
        coeffk = _safe(upper_tri_coeffs_for_half_quadratic(symmetrize(A_list[k]))[sel_idx])
        for q_id in np.where(np.abs(coeffk) > cfg.tol)[0]:
            qc_src.append(int(q_id))
            qc_dst.append(int(k - 1))
            qc_attr.append([float(coeffk[q_id])])

    ei, ea = make_edge_tensors(qc_src, qc_dst, qc_attr)
    data["quad", "qcoef", "constr"].edge_index = ei
    data["quad", "qcoef", "constr"].edge_attr = ea
    rev = torch.stack([ei[1], ei[0]], 0) if ei.numel() > 0 else torch.empty((2, 0), dtype=torch.long)
    data["constr", "rev_qcoef", "quad"].edge_index = rev
    data["constr", "rev_qcoef", "quad"].edge_attr = ea

    # var -> constr
    vc_src, vc_dst, vc_attr = [], [], []
    for k in range(1, m_total):
        bk = _safe(b_list[k].astype(float))
        keep = (
            np.arange(n)
            if (cfg.max_lin_edges_per_constr is None or cfg.max_lin_edges_per_constr >= n)
            else np.argpartition(np.abs(bk), -cfg.max_lin_edges_per_constr)[-cfg.max_lin_edges_per_constr:]
        )
        for i in keep:
            val = float(bk[int(i)])
            if abs(val) > cfg.tol:
                vc_src.append(int(i))
                vc_dst.append(int(k - 1))
                vc_attr.append([val])

    ei, ea = make_edge_tensors(vc_src, vc_dst, vc_attr)
    data["var", "lin", "constr"].edge_index = ei
    data["var", "lin", "constr"].edge_attr = ea
    rev = torch.stack([ei[1], ei[0]], 0) if ei.numel() > 0 else torch.empty((2, 0), dtype=torch.long)
    data["constr", "rev_lin", "var"].edge_index = rev
    data["constr", "rev_lin", "var"].edge_attr = ea

    data.global_x = torch.tensor(
        [[_safe_scalar(c_list[0]), float(n) / 500.0, float(m_constr) / 500.0]],
        dtype=torch.float,
    )

    delta, choice_mask, finite_mask, target_case, delta_for_weight = make_regression_target_from_row(
        row,
        epsilon_gap=cfg.epsilon_gap,
        delta_transform=cfg.delta_transform,
    )

    data.delta = torch.tensor([float(delta)], dtype=torch.float)
    data.choice_mask = torch.tensor([float(choice_mask)], dtype=torch.float)
    data.finite_mask = torch.tensor([float(finite_mask)], dtype=torch.float)
    data.delta_raw_for_weight = torch.tensor(
        [float(np.nan_to_num(delta_for_weight, nan=0.0))],
        dtype=torch.float,
    )
    data.target_case = target_case
    data.row_key = to_key(row.get("id_global", row.get("id", row.get("__match_key__", "NA"))))
    return data


# ============================================================
# 10) Dataset
# ============================================================

def build_dataframe_from_instance_results(instances) -> pd.DataFrame:
    rows = []
    for inst in instances:
        res = get_field(inst, "results", {}) or {}
        if not isinstance(res, dict):
            res = {}
        rows.append({
            "instance_id": to_key(get_field(inst, "instance_id")),
            "group_id": to_key(get_field(inst, "group_id")),
            "size_tag": to_key(get_field(inst, "size_tag")),
            "id": to_key(get_field(inst, "instance_id")),
            "lp_obj": res.get("lp_obj", np.nan),
            "lp_status": res.get("lp_status", ""),
            "sdp_obj": res.get("sdp_obj", np.nan),
            "sdp_status": res.get("sdp_status", ""),
        })
    df = pd.DataFrame(rows)
    if (df[["lp_obj", "sdp_obj"]].notna().sum().sum() == 0 and (df["lp_status"].astype(str) == "").all()):
        raise RuntimeError("No CSV and instances[].results has no LP/SDP labels.")
    return df


class QCQPTripartiteDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        pkl_path,
        csv_path,
        cfg: GraphConfig,
        cache_dir=None,
        max_items=None,
        require_labels=True,
        filter_size_tag=None,
        filter_n=None,
        filter_m_total=None,
        verbose=True,
        allow_row_order_fallback=True,
    ):
        super().__init__()
        self.cfg = cfg
        self.cache_dir = cache_dir
        self.verbose = verbose
        if cache_dir:
            os.makedirs(cache_dir, exist_ok=True)

        self.instances = load_instances_pkl(pkl_path)

        if csv_path and os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
        elif require_labels:
            df = build_dataframe_from_instance_results(self.instances)
        else:
            df = pd.DataFrame()

        self.candidate_keys = ["id_global", "id", "instance_id", "group_id", "size_tag"]
        self.instances_by_key: Dict[str, Any] = {}
        for idx, inst in enumerate(self.instances):
            for kn in self.candidate_keys:
                k = to_key(get_field(inst, kn))
                if k is not None:
                    self.instances_by_key[k] = inst
            self.instances_by_key[f"__rowidx__{idx}"] = inst

        if df.shape[0] == 0:
            df = pd.DataFrame([
                {
                    "__match_key__": f"__rowidx__{i}",
                    "lp_obj": np.nan,
                    "lp_status": "",
                    "sdp_obj": np.nan,
                    "sdp_status": "",
                }
                for i in range(len(self.instances))
            ])

        def row_key_match(row):
            for k in self.candidate_keys:
                v = to_key(row.get(k))
                if v is not None and v in self.instances_by_key:
                    return v
            return None

        match = [row_key_match(row) for _, row in df.iterrows()]
        keep_mask = [m is not None for m in match]
        matched_count = int(np.sum(keep_mask))

        if matched_count == 0 and allow_row_order_fallback:
            if len(df) == len(self.instances):
                if self.verbose:
                    print(f"[{_ts()}] [match] No key matches; ROW-ORDER fallback.")
                df = df.copy()
                df["__match_key__"] = [f"__rowidx__{i}" for i in range(len(df))]
            else:
                raise RuntimeError(
                    f"No rows matched and row-order fallback impossible "
                    f"(df={len(df)}, instances={len(self.instances)})."
                )
        else:
            df = df[np.array(keep_mask, dtype=bool)].copy()
            df["__match_key__"] = [m for m in match if m is not None]
            df.reset_index(drop=True, inplace=True)
            if self.verbose:
                print(f"[{_ts()}] [match] matched rows: {len(df)}/{len(match)}")

        if filter_size_tag is not None:
            df = df[df.get("size_tag", "").astype(str) == str(filter_size_tag)].copy()
            df.reset_index(drop=True, inplace=True)

        if filter_n is not None or filter_m_total is not None:
            ok_rows = []
            for i in range(len(df)):
                inst = self.instances_by_key[str(df.loc[i, "__match_key__"])]
                n_i = int(get_field(inst, "n", -1))
                m_i = int(get_field(inst, "m", -1))
                ok = ((filter_n is None or n_i == filter_n) and (filter_m_total is None or m_i == filter_m_total))
                ok_rows.append(ok)
            df = df[np.array(ok_rows, dtype=bool)].copy()
            df.reset_index(drop=True, inplace=True)

        dd_list, mm_list, fm_list, cc_list, dw_list = [], [], [], [], []
        for _, row in df.iterrows():
            d, m, f, c, dw = make_regression_target_from_row(
                row,
                epsilon_gap=self.cfg.epsilon_gap,
                delta_transform=self.cfg.delta_transform,
            )
            dd_list.append(d)
            mm_list.append(m)
            fm_list.append(f)
            cc_list.append(c)
            dw_list.append(dw)

        df["__delta__"] = dd_list
        df["__choice_mask__"] = mm_list
        df["__finite_mask__"] = fm_list
        df["__target_case__"] = cc_list
        df["__delta_raw_for_weight__"] = dw_list

        case_counts = pd.Series(cc_list).value_counts().to_dict()
        total = len(cc_list)
        n_active = int(np.sum(mm_list))
        n_finite = int(np.sum(fm_list))

        print(f"[{_ts()}] [target] case counts (N={total}): {case_counts}")
        print(f"[{_ts()}] [target] active rows: {n_active}/{total} ({100 * n_active / max(1, total):.1f}%)")
        print(f"[{_ts()}] [target] finite rows for MSE: {n_finite}/{total} ({100 * n_finite / max(1, total):.1f}%)")

        if max_items and max_items > 0:
            df = df.iloc[:max_items].copy()
            df.reset_index(drop=True, inplace=True)

        self.df = df

    def __len__(self):
        return len(self.df)

    def _cache_path(self, idx, row):
        return os.path.join(
            self.cache_dir,
            f"graph_{idx:08d}_{safe_filename(str(row['__match_key__']))}.pt",
        )

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.cache_dir:
            cp = self._cache_path(idx, row)
            if os.path.exists(cp):
                return torch.load(cp)
        inst = self.instances_by_key[str(row["__match_key__"])]
        g = build_tripartite_term_graph(inst, row, self.cfg)
        if self.cache_dir:
            torch.save(g, cp)
        return g


# ============================================================
# 11) Model — single regression head
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


def edge_weighted_sum(x_src, edge_index, edge_attr, num_dst):
    if edge_index.numel() == 0:
        return x_src.new_zeros((num_dst, x_src.size(1)))
    msg = x_src[edge_index[0]] * edge_attr
    return scatter(msg, edge_index[1], dim=0, dim_size=num_dst, reduce="sum")


class TripartiteTermGNN(nn.Module):
    def __init__(self, hidden=64, layers=3, dropout=0.1):
        super().__init__()
        self.hidden = hidden
        self.layers = layers
        self.dropout = dropout
        H = hidden

        self.lin_var = nn.Linear(5, H)
        self.lin_quad = nn.Linear(5, H)
        self.lin_con = nn.Linear(2, H)

        self.f_v_to_q = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_v_to_c = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_q_to_c = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_c_to_q = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_c_to_v = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_q_to_v = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])

        self.g_q1 = nn.ModuleList([MLP(H * 2, H) for _ in range(layers)])
        self.g_c = nn.ModuleList([MLP(H * 3, H) for _ in range(layers)])
        self.g_q2 = nn.ModuleList([MLP(H * 2, H) for _ in range(layers)])
        self.g_v = nn.ModuleList([MLP(H * 3, H) for _ in range(layers)])

        head_in = H * 3 + 3
        self.head_delta = nn.Sequential(
            nn.Linear(head_in, H),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(H, 1),
        )

    def forward(self, data):
        h_v = self.lin_var(data["var"].x)
        h_q = self.lin_quad(data["quad"].x)
        h_c = self.lin_con(data["constr"].x)

        e_vq, w_vq = data["var", "inc", "quad"].edge_index, data["var", "inc", "quad"].edge_attr
        e_vc, w_vc = data["var", "lin", "constr"].edge_index, data["var", "lin", "constr"].edge_attr
        e_qc, w_qc = data["quad", "qcoef", "constr"].edge_index, data["quad", "qcoef", "constr"].edge_attr
        e_cq, w_cq = data["constr", "rev_qcoef", "quad"].edge_index, data["constr", "rev_qcoef", "quad"].edge_attr
        e_cv, w_cv = data["constr", "rev_lin", "var"].edge_index, data["constr", "rev_lin", "var"].edge_attr
        e_qv, w_qv = data["quad", "rev_inc", "var"].edge_index, data["quad", "rev_inc", "var"].edge_attr

        nv = data["var"].num_nodes
        nq = data["quad"].num_nodes
        nc = data["constr"].num_nodes

        for t in range(self.layers):
            h_q_bar = self.g_q1[t](torch.cat([
                h_q,
                edge_weighted_sum(self.f_v_to_q[t](h_v), e_vq, w_vq, nq),
            ], 1))
            h_c_new = self.g_c[t](torch.cat([
                h_c,
                edge_weighted_sum(self.f_v_to_c[t](h_v), e_vc, w_vc, nc),
                edge_weighted_sum(self.f_q_to_c[t](h_q_bar), e_qc, w_qc, nc),
            ], 1))
            h_q_new = self.g_q2[t](torch.cat([
                h_q_bar,
                edge_weighted_sum(self.f_c_to_q[t](h_c_new), e_cq, w_cq, nq),
            ], 1))
            h_v_new = self.g_v[t](torch.cat([
                h_v,
                edge_weighted_sum(self.f_c_to_v[t](h_c_new), e_cv, w_cv, nv),
                edge_weighted_sum(self.f_q_to_v[t](h_q_new), e_qv, w_qv, nv),
            ], 1))

            h_v = F.dropout(F.relu(h_v_new), p=self.dropout, training=self.training)
            h_q = F.dropout(F.relu(h_q_new), p=self.dropout, training=self.training)
            h_c = F.dropout(F.relu(h_c_new), p=self.dropout, training=self.training)

        B = int(data["var"].batch.max().item()) + 1
        pv = global_mean_pool(h_v, data["var"].batch, size=B)
        pq = global_mean_pool(h_q, data["quad"].batch, size=B) if nq > 0 else h_v.new_zeros((B, self.hidden))
        pc = global_mean_pool(h_c, data["constr"].batch, size=B) if nc > 0 else h_v.new_zeros((B, self.hidden))

        gx = data.global_x
        if gx.dim() == 1:
            gx = gx.view(-1, 3)
        if gx.size(0) != B:
            gx = gx.view(B, -1)

        g = torch.cat([pv, pq, pc, gx], dim=1)
        return self.head_delta(g).view(-1)


# ============================================================
# 12) Metrics — LP-biased throughout
# ============================================================

def safe_pearson_r(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    m = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[m]
    yp = yp[m]
    if len(yt) == 0 or np.std(yt) <= 0 or np.std(yp) <= 0:
        return float("nan")
    r = np.corrcoef(yt, yp)[0, 1]
    return float(r) if np.isfinite(r) else float("nan")


def r_accuracy(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 0.1) -> float:
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    m = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[m]
    yp = yp[m]
    if len(yt) == 0:
        return float("nan")
    true_lp = yt > -eps
    pred_lp = yp > -eps
    return float(np.mean(true_lp == pred_lp))


def sign_accuracy_tau(y_true: np.ndarray, y_pred: np.ndarray, tau: float = 0.1) -> Tuple[float, float]:
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp) & (np.abs(yt) >= tau)
    if mask.sum() == 0:
        return float("nan"), 0.0
    true_lp = yt[mask] > -tau
    pred_lp = yp[mask] > -tau
    return float(np.mean(true_lp == pred_lp)), float(mask.mean())


# ============================================================
# 13) Sample weights
# ============================================================

def make_sample_weight_tensor(delta_for_weight: torch.Tensor, alpha: float) -> torch.Tensor:
    a = float(np.clip(alpha, 0.0, 1.0))
    strength = torch.nan_to_num(delta_for_weight.abs(), nan=0.0, posinf=0.0, neginf=0.0)
    return (1.0 - a) + a * strength


# ============================================================
# 14) Train / Eval
# ============================================================

def train_one_epoch(
    model,
    loader,
    opt,
    device,
    use_sample_weights: bool = True,
    weight_alpha: float = 0.30,
) -> float:
    model.train()
    total_loss = 0.0
    total_n = 0

    for batch in loader:
        batch = batch.to(device)
        opt.zero_grad()

        lk = model(batch)
        d = batch.delta.view(-1).float()
        active = batch.choice_mask.view(-1).float()
        dsw = batch.delta_raw_for_weight.view(-1).float()

        lk_safe = torch.where(torch.isfinite(lk), lk, torch.zeros_like(lk))
        n_active = active.sum()
        if n_active == 0:
            continue

        if use_sample_weights:
            w = make_sample_weight_tensor(dsw, weight_alpha)
            loss = ((lk_safe - d) ** 2 * active * w).sum() / ((active * w).sum() + 1e-12)
        else:
            loss = ((lk_safe - d) ** 2 * active).sum() / (n_active + 1e-12)

        if not torch.isfinite(loss):
            opt.zero_grad()
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        total_loss += float(loss.item()) * int(n_active.item())
        total_n += int(n_active.item())

    return total_loss / max(1, total_n)


@torch.no_grad()
def evaluate_full(model, loader, device, tau_sign: float = 0.1) -> Tuple[dict, dict, np.ndarray, np.ndarray]:
    model.eval()

    d_all, dk_all, active_all, finite_all, case_all, keys_all = [], [], [], [], [], []

    for batch in loader:
        batch = batch.to(device)
        lk = model(batch)
        d = batch.delta.view(-1).float()
        active = batch.choice_mask.view(-1).float()
        finite = batch.finite_mask.view(-1).float()
        lk_safe = torch.where(torch.isfinite(lk), lk, torch.zeros_like(lk))

        d_all.append(d.cpu())
        dk_all.append(lk_safe.cpu())
        active_all.append(active.cpu())
        finite_all.append(finite.cpu())

        if hasattr(batch, "target_case"):
            tc = batch.target_case
            case_all.extend(tc if isinstance(tc, (list, tuple)) else [tc] * int(d.numel()))
        else:
            case_all.extend(["unknown"] * int(d.numel()))

        if hasattr(batch, "row_key"):
            rk = batch.row_key
            keys_all.extend([str(k) for k in rk] if isinstance(rk, (list, tuple)) else [str(rk)] * int(d.numel()))
        else:
            keys_all.extend(["NA"] * int(d.numel()))

    d = torch.cat(d_all).numpy()
    dk = torch.cat(dk_all).numpy()
    active = torch.cat(active_all).numpy()
    finite = torch.cat(finite_all).numpy()
    case_arr = np.array(case_all)

    active_idx = active == 1
    finite_idx = finite == 1

    d_active = d[active_idx]
    dk_active = dk[active_idx]
    d_finite = d[finite_idx]
    dk_finite = dk[finite_idx]

    metrics = {
        "coverage_active": float(np.mean(active)),
        "coverage_finite": float(np.mean(finite)),
    }

    if finite_idx.sum() == 0:
        metrics["mse"] = float("nan")
        metrics["mae"] = float("nan")
        metrics["pearson_r"] = float("nan")
    else:
        metrics["mse"] = float(mean_squared_error(d_finite, dk_finite))
        metrics["mae"] = float(mean_absolute_error(d_finite, dk_finite))
        metrics["pearson_r"] = safe_pearson_r(d_finite, dk_finite)

    if active_idx.sum() == 0:
        metrics["r_accuracy"] = float("nan")
        metrics["sign_acc_tau"] = float("nan")
        metrics["sign_coverage"] = 0.0
    else:
        metrics["r_accuracy"] = r_accuracy(d_active, dk_active, eps=tau_sign)
        sacc, scov = sign_accuracy_tau(d_active, dk_active, tau=tau_sign)
        metrics["sign_acc_tau"] = sacc
        metrics["sign_coverage"] = scov

    for case in ["finite", "sdp_unbounded", "lp_unbounded", "excluded"]:
        cidx = (case_arr == case) & active_idx
        metrics[f"n_{case}"] = int(cidx.sum())

    preds = {
        "row_key": keys_all,
        "delta_true": d.tolist(),
        "delta_pred": dk.tolist(),
        "choice_mask": active.tolist(),
        "finite_mask": finite.tolist(),
        "target_case": case_all,
    }

    return metrics, preds, d_active, dk_active


# ============================================================
# 15) Deterministic split
# ============================================================

def make_deterministic_splits(keys: pd.Series, train: float = 0.8, val: float = 0.1):
    h = pd.util.hash_pandas_object(keys.astype(str), index=False).to_numpy(dtype=np.uint64)
    b = (h % 100).astype(np.int64)
    tc = int(train * 100)
    vc = int((train + val) * 100)
    return (
        np.where(b < tc)[0],
        np.where((b >= tc) & (b < vc))[0],
        np.where(b >= vc)[0],
    )


def pick_split_key(df: pd.DataFrame) -> pd.Series:
    for col in ["id_global", "id", "instance_id", "group_id", "size_tag", "__match_key__"]:
        if col in df.columns and df[col].notna().any():
            return df[col].astype(str)
    return pd.Series(np.arange(len(df)), index=df.index).astype(str)


# ============================================================
# 16) Output helpers
# ============================================================

def save_scatter(d_true, d_pred, run_dir, fname, title):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(d_true, d_pred, s=10, alpha=0.45, edgecolors="none")
    lo = min(d_true.min(), d_pred.min())
    hi = max(d_true.max(), d_pred.max())
    ax.plot([lo, hi], [lo, hi], "r--", label="Perfect")
    ax.axhline(0, ls="--", color="gray", alpha=0.4)
    ax.axvline(0, ls="--", color="gray", alpha=0.4)
    ax.axhline(-0.1, ls=":", color="blue", alpha=0.4, label="-tau")
    ax.axvline(-0.1, ls=":", color="blue", alpha=0.4)
    ax.set_xlabel("True delta / target")
    ax.set_ylabel("Predicted delta")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.savefig(run_dir / fname, dpi=200)
    plt.close()
    print(f"[{_ts()}] [save] {fname}")


def delta_to_lp_decision(delta: np.ndarray, tau: float) -> np.ndarray:
    x = np.asarray(delta, dtype=float)
    return (x > -tau).astype(int)


def save_sign_confusion_matrix(d_true, d_pred, tau, run_dir, fname, title):
    yt = np.asarray(d_true, dtype=float)
    yp = np.asarray(d_pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[mask], yp[mask]

    true_dec = delta_to_lp_decision(yt, tau)
    pred_dec = delta_to_lp_decision(yp, tau)

    cm = confusion_matrix(true_dec, pred_dec, labels=[0, 1])
    acc = float(np.mean(true_dec == pred_dec)) if len(true_dec) > 0 else float("nan")

    pd.DataFrame(
        cm,
        index=["true_SDP(0)", "true_LP(1)"],
        columns=["pred_SDP(0)", "pred_LP(1)"],
    ).to_csv(run_dir / fname.replace(".png", ".csv"), index=True)

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=["SDP(0)", "LP(1)"]).plot(ax=ax)
    ax.set_title(f"{title}  acc={acc:.3f}  tau={tau:.3f}")
    plt.tight_layout()
    plt.savefig(run_dir / fname, dpi=200)
    plt.close()
    print(f"[{_ts()}] [save] {fname}")


def save_epoch_log(epoch_log, run_dir, tag):
    pd.DataFrame(epoch_log).to_csv(run_dir / f"epoch_log_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] epoch_log_{tag}.csv")


def save_results(metrics, run_dir, tag, split, cfg):
    row = {
        "split": split,
        "run_tag": tag,
        **metrics,
        "train_pkl": cfg.train_pkl,
        "test_pkl": cfg.test_pkl or "holdout",
        "epochs": cfg.epochs,
        "hidden": cfg.hidden,
        "layers": cfg.layers,
        "dropout": cfg.dropout,
        "lr": cfg.lr,
        "epsilon_gap": cfg.epsilon_gap,
        "delta_transform": cfg.delta_transform,
        "tau_sign": cfg.tau_sign,
        "use_sample_weights": cfg.use_sample_weights,
        "weight_alpha": cfg.weight_alpha,
        "gap_method": "fixed_additive",
        "decision_rule": "lp_biased",
        "unbounded_handling": "kept_in_training_finite_mse_only",
    }
    pd.DataFrame([row]).to_csv(run_dir / f"results_reg_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] results_reg_{tag}.csv")


def save_predictions(preds, run_dir, tag, split):
    df = pd.DataFrame(preds)
    df.insert(0, "split", split)
    df.to_csv(run_dir / f"preds_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] preds_{tag}.csv")


def save_config(cfg, run_dir, tag):
    pd.DataFrame([asdict(cfg)]).to_csv(run_dir / f"config_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] config_{tag}.csv")


# ============================================================
# 17) main()
# ============================================================

def main(run_cfg: Optional[RunConfig] = None):
    if run_cfg is None:
        run_cfg = RunConfig()

    np.random.seed(0)
    torch.manual_seed(0)

    run_dir, tag = _make_run_dir(run_cfg)

    print(f"[{_ts()}] ══════════════════════════════════════════════════")
    print(f"[{_ts()}] GNN REGRESSOR (keep unbounded)  tag={tag}")
    print(f"[{_ts()}] Output dir   : {run_dir.resolve()}")
    print(f"[{_ts()}] Gap formula  : FIXED ADDITIVE")
    print(f"[{_ts()}] delta = (lp - sdp) / (|lp| + |sdp| + epsilon_gap)")
    print(f"[{_ts()}] epsilon_gap={run_cfg.epsilon_gap}")
    print(f"[{_ts()}] delta_transform={run_cfg.delta_transform}  tau_sign={run_cfg.tau_sign}")
    print(f"[{_ts()}] use_sample_weights={run_cfg.use_sample_weights}  weight_alpha={run_cfg.weight_alpha}")
    print(f"[{_ts()}] Unbounded handling: kept with targets ±1.0; excluded from finite MSE only")
    print(f"[{_ts()}] Metrics: LP-biased (delta > -tau)")
    print(f"[{_ts()}] Split (holdout): 72/8/20 — test=same 20% as ML")
    print(f"[{_ts()}] ══════════════════════════════════════════════════")

    gcfg = GraphConfig(
        tol=run_cfg.tol,
        epsilon_gap=run_cfg.epsilon_gap,
        delta_transform=run_cfg.delta_transform,
        max_quad_nodes=(None if run_cfg.max_quad_nodes <= 0 else int(run_cfg.max_quad_nodes)),
        max_lin_edges_per_constr=(None if run_cfg.max_lin_edges_per_constr <= 0 else int(run_cfg.max_lin_edges_per_constr)),
    )

    mit = None if run_cfg.max_items_train <= 0 else int(run_cfg.max_items_train)
    mie = None if run_cfg.max_items_test <= 0 else int(run_cfg.max_items_test)

    train_ds_full = QCQPTripartiteDataset(
        run_cfg.train_pkl,
        run_cfg.train_csv,
        gcfg,
        cache_dir=run_cfg.cache_dir,
        max_items=mit,
        require_labels=True,
        filter_size_tag=run_cfg.train_size_tag,
        filter_n=run_cfg.train_n,
        filter_m_total=run_cfg.train_m_total,
        verbose=run_cfg.verbose,
        allow_row_order_fallback=run_cfg.allow_row_order_fallback,
    )

    if len(train_ds_full) == 0:
        raise RuntimeError("Train dataset empty.")

    if run_cfg.test_pkl:
        tr_idx, va_idx, _ = make_deterministic_splits(pick_split_key(train_ds_full.df), 0.9, 0.1)
        test_ds = QCQPTripartiteDataset(
            run_cfg.test_pkl,
            run_cfg.test_csv,
            gcfg,
            cache_dir=None,
            max_items=mie,
            require_labels=True,
            filter_size_tag=run_cfg.test_size_tag,
            filter_n=run_cfg.test_n,
            filter_m_total=run_cfg.test_m_total,
            verbose=run_cfg.verbose,
            allow_row_order_fallback=run_cfg.allow_row_order_fallback,
        )
        if len(test_ds) == 0:
            raise RuntimeError("Test dataset empty.")
        test_split_name = "external_test"
        print(f"[{_ts()}] [split] External: train=90% val=10% of training PKL", flush=True)
    else:
        tr_idx, va_idx, te_idx = make_deterministic_splits(pick_split_key(train_ds_full.df), 0.72, 0.08)
        test_ds = Subset(train_ds_full, te_idx.tolist())
        test_split_name = "holdout_test"
        print(
            f"[{_ts()}] [split] Holdout: train={len(tr_idx)} val={len(va_idx)} "
            f"test={len(te_idx)} (bucket>=80, same as ML)",
            flush=True,
        )

    train_ds = Subset(train_ds_full, tr_idx.tolist())
    val_ds = Subset(train_ds_full, va_idx.tolist())

    print(f"[{_ts()}] [data] train={len(train_ds)} | val={len(val_ds)} | test={len(test_ds)}")

    train_loader = DataLoader(train_ds, batch_size=run_cfg.batch_train, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=run_cfg.batch_eval, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=run_cfg.batch_eval, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[{_ts()}] [device] {device}")

    model = TripartiteTermGNN(run_cfg.hidden, run_cfg.layers, run_cfg.dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=run_cfg.lr, weight_decay=run_cfg.weight_decay)

    epoch_log = []
    for epoch in range(1, run_cfg.epochs + 1):
        loss = train_one_epoch(
            model,
            train_loader,
            opt,
            device,
            use_sample_weights=run_cfg.use_sample_weights,
            weight_alpha=run_cfg.weight_alpha,
        )
        vm, _, _, _ = evaluate_full(model, val_loader, device, tau_sign=run_cfg.tau_sign)
        row = {"epoch": epoch, "train_loss": loss, **{f"val_{k}": v for k, v in vm.items()}}
        epoch_log.append(row)

        print(
            f"[{_ts()}] Epoch {epoch:02d} | loss={loss:.6f} | "
            f"mse_finite={vm['mse']:.6f} | mae_finite={vm['mae']:.4f} | "
            f"r_finite={vm['pearson_r']:.3f} | "
            f"r_acc={vm['r_accuracy']:.3f} | "
            f"sign@tau={vm['sign_acc_tau']:.3f} (cov={vm['sign_coverage']:.2f}) | "
            f"n_finite={vm.get('n_finite', 0)} "
            f"n_sdp_unb={vm.get('n_sdp_unbounded', 0)} "
            f"n_lp_unb={vm.get('n_lp_unbounded', 0)}",
            flush=True,
        )

    save_epoch_log(epoch_log, run_dir, tag)

    test_metrics, preds, d_true, d_pred = evaluate_full(
        model,
        test_loader,
        device,
        tau_sign=run_cfg.tau_sign,
    )

    print(f"\n[{_ts()}] TEST METRICS:")
    for k, v in test_metrics.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    save_predictions(preds, run_dir, tag, test_split_name)
    save_scatter(d_true, d_pred, run_dir, f"scatter_{tag}.png", "GNN Regressor — actual vs predicted target")
    save_sign_confusion_matrix(
        d_true,
        d_pred,
        run_cfg.tau_sign,
        run_dir,
        f"sign_cm_{tag}.png",
        "GNN Regression -> LP/SDP decision",
    )
    save_results(test_metrics, run_dir, tag, test_split_name, run_cfg)
    save_config(run_cfg, run_dir, tag)

    g0 = train_ds_full[0]
    print(f"\n[{_ts()}] [graph sanity]")
    print(f"  var    : {g0['var'].num_nodes} nodes  shape={list(g0['var'].x.shape)}")
    print(f"  quad   : {g0['quad'].num_nodes} nodes  shape={list(g0['quad'].x.shape)}")
    print(f"  constr : {g0['constr'].num_nodes} nodes  shape={list(g0['constr'].x.shape)}")
    for et in g0.edge_types:
        print(f"  {et}: E={g0[et].edge_index.shape[1]}  attr={list(g0[et].edge_attr.shape)}")
    print(
        f"  delta={float(g0.delta):.4f}  "
        f"choice_mask={int(g0.choice_mask)}  "
        f"finite_mask={int(g0.finite_mask)}  "
        f"target_case={g0.target_case}  "
        f"delta_raw_for_weight={float(g0.delta_raw_for_weight):.4f}"
    )

    written = sorted(p.name for p in run_dir.glob("*"))
    print(f"\n[{_ts()}] Files written ({len(written)}): {written}")

    return model, test_metrics


if __name__ == "__main__":
    cfg = RunConfig(
        delta_transform="none",
        tau_sign=0.1,
        use_sample_weights=True,
        weight_alpha=0.30,
    )
    model, metrics = main(cfg)